# FaceProof 03 — probes + cross-generator (C1, C4)

**Day 2 gate (part 2) + Day 3 (RQ1).** Train logistic probes on frozen CLIP (C4) and ResNet (C1)
features; measure the drop from in-distribution (held-out StyleGAN) to an unseen generator (SD).
**H1:** the generalization gap is smaller for CLIP. Probes are saved for nb 04/06.

## 1. Setup

In [ ]:
REPO_URL = "https://github.com/Ezed9/faceProof.git"
BRANCH   = "main"
!git clone -b $BRANCH $REPO_URL
%cd faceProof
!pip install -q scikit-learn joblib pyyaml
import sys, os
sys.path.append(os.getcwd())

In [ ]:
from google.colab import drive
from pathlib import Path
import numpy as np, pandas as pd
drive.mount('/content/drive')
ROOT          = Path('/content/drive/MyDrive/faceproof')
CROPS_ROOT    = ROOT / 'data' / 'crops'
FEATURES_ROOT = ROOT / 'models' / 'features'
PROBES_ROOT   = ROOT / 'models' / 'probes'
REPORTS_ROOT  = ROOT / 'reports'
MANIFEST      = ROOT / 'data' / 'manifest.csv'
RESULTS_CSV   = REPORTS_ROOT / 'results.csv'
(REPORTS_ROOT / 'figures').mkdir(parents=True, exist_ok=True)
PROBES_ROOT.mkdir(parents=True, exist_ok=True)

In [ ]:
import csv
RESULT_FIELDS = ['condition','train_gen','test_gen','corruption','strength','seed','metric','value']
def log_result(**row):
    full = {k: row.get(k, '') for k in RESULT_FIELDS}
    new = not RESULTS_CSV.exists()
    with open(RESULTS_CSV, 'a', newline='') as f:
        w = csv.DictWriter(f, fieldnames=RESULT_FIELDS)
        if new: w.writeheader()
        w.writerow(full)

## 2. Load manifest + both cached feature sets

In [ ]:
df = pd.read_csv(MANIFEST)
X = {'c4_clip': np.load(FEATURES_ROOT/'clip_all.npy'),
     'c1_resnet': np.load(FEATURES_ROOT/'resnet_all.npy')}
for k,v in X.items():
    assert len(v)==len(df), f'{k} rows must match manifest (re-run nb 02)'
print('rows:', len(df)); print(df['split'].value_counts())

## 3. Train both probes (on `train` only) and save them

Probe hyperparameters come from `configs/model.yaml`.

In [ ]:
import yaml, joblib
from src.probe import train_probe, predict_proba
pcfg = yaml.safe_load(open('configs/model.yaml'))['probe']
m_train = (df['split']=='train').values
y = df['label'].values
probes={}
for cond, feats in X.items():
    clf = train_probe(feats[m_train], y[m_train], C=pcfg['C'], max_iter=pcfg['max_iter'])
    joblib.dump(clf, PROBES_ROOT/f'{cond}.joblib'); probes[cond]=clf
    print('trained+saved', cond)

## 4. Evaluate: in-distribution vs cross-generator

> **Why these masks:** the unseen SD rows live in `split=='test'` and are *all fake*, so AUROC on that split alone is undefined. Each eval group pairs the held-out **reals** (label 0 in `test_indist`) with the fakes of interest: in-dist = StyleGAN, cross-gen = Stable Diffusion.

In [ ]:
from src.metrics import auroc, eer
real_test = (df['label']==0) & (df['split']=='test_indist')
GROUPS = {'stylegan_indist': (df['split']=='test_indist').values,
          'stable_diffusion': (real_test | (df['generator']=='stable_diffusion')).values}
for g,m in GROUPS.items():
    yy=y[m]; assert yy.min()==0 and yy.max()==1, f'{g} must contain both classes'
    print(g, 'n=', int(m.sum()), 'real=', int((yy==0).sum()), 'fake=', int((yy==1).sum()))

def evaluate(clf, feats, mask):
    p=predict_proba(clf, feats[mask]); yy=y[mask]
    return auroc(yy,p), eer(yy,p)

rows=[]
for cond,feats in X.items():
    for tg,m in GROUPS.items():
        au,er=evaluate(probes[cond],feats,m)
        log_result(condition=cond,train_gen='stylegan',test_gen=tg,corruption='none',seed=13,metric='AUROC',value=round(au,4))
        log_result(condition=cond,train_gen='stylegan',test_gen=tg,corruption='none',seed=13,metric='EER',value=round(er,4))
        rows.append({'condition':cond,'test_gen':tg,'AUROC':round(au,4),'EER':round(er,4)})
res=pd.DataFrame(rows); print(res.to_string(index=False))

## 5. ✅ Gates — in-dist AUROC (Day 2) + generalization-gap table (Day 3)

In [ ]:
pivot=res.pivot(index='condition',columns='test_gen',values='AUROC')
pivot['gap']=pivot['stylegan_indist']-pivot['stable_diffusion']
print(pivot.round(4).to_string())
indist_ok=(pivot['stylegan_indist']>0.90).all()
print('\n✅ Day 2 gate:' if indist_ok else '⚠️ Day 2 gate:',
      'in-dist AUROC > 0.90 for both' if indist_ok else 'some in-dist AUROC <= 0.90')
print('✅ H1 directional: CLIP gap <= ResNet gap' if pivot.loc['c4_clip','gap']<=pivot.loc['c1_resnet','gap']
      else '⚠️ H1 not supported this run: report honestly')

---
**Log it** in `experiments.md`. **Day 4 next:** robustness.